# Transaction Anomaly Detection

*Exploratory Transaction Analysis · Temporal Validation · Risk Screening*

---

Transaction screening models must recognize unusual activity without overwhelming investigators with false alarms. This study lays out an end-to-end analysis framework emphasizing outcome imbalance, time-aware evaluation, score quality, and alert workload.

**Research objective:** assess whether a transaction model can distinguish labeled anomalies using only information available at decision time.

**Dataset requirement:** the analytic cells require an actual, rights-cleared labeled transaction dataset matching the documented schema. No such dataset is bundled and no real-data performance figures are currently reported.

### Study roadmap

1. Research question and data
2. Exploratory data analysis and quality checks
3. Feature relationships and modeling
4. Model evaluation and visual diagnostics
5. Interpretation, limitations, and conclusions

---

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "README.md").exists() and (ROOT.parent / "README.md").exists():
    ROOT = ROOT.parent
if not (ROOT / "README.md").exists():
    raise RuntimeError("Run the notebook from the project root or notebooks/ directory")
sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)

## 1. Real dataset requirements
Read the exact input schema and provenance requirements before preparing a CSV. **Do not rename a different dataset to fit the filename**.

In [ ]:
print((ROOT/"docs"/"DATA.md").read_text(encoding="utf-8")[:8000])

## 2. Optional exploratory data analysis on an ACTUAL, user-provided dataset
This cell fails safely if the dataset is missing; there is no synthetic fallback. The CSV must meet the contract in `docs/DATA.md`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
path = ROOT / "data" / "real" / "transactions.csv"
if not path.exists():
    raise FileNotFoundError(f"Real labeled dataset not provided: {path}. Skip experiment until verified.")
df = pd.read_csv(path)
expected={"transaction_id","timestamp","amount","international","card_present","account_age_days","is_anomaly"}
assert expected.issubset(df.columns), f"Missing columns: {sorted(expected-set(df.columns))}"
display(df.head())
display(df.isna().mean().sort_values(ascending=False))
display(df["is_anomaly"].value_counts(normalize=True).rename("class_fraction"))
df["timestamp"]=pd.to_datetime(df["timestamp"],utc=True,errors="raise")
df.sort_values("timestamp")["amount"].plot(figsize=(10,3),title="Transaction amount over time")
plt.show()

## 3. Temporal train, validation, and test
Use time cutoffs that correspond to the **actual real dataset**, with strictly forward validation/test. Inspect the modular code below; do not invent cutoffs or result claims.

In [ ]:
from transaction_anomaly import data, evaluation, pipeline
import inspect
print("Core modules:")
for module in (data,evaluation,pipeline):
    funcs=[n for n,v in vars(module).items() if inspect.isfunction(v) and getattr(v,"__module__","")==module.__name__]
    print(module.__name__,funcs[:25])

## 4. Running a real-data benchmark
After confirming the real dataset's provenance and date range, use `python -m transaction_anomaly.cli train --help` to select real cutoffs, then run from a terminal with `PYTHONPATH=src`. The pipeline compares prevalence and logistic baselines and reports average precision, recall, F1, calibration, and drift. This notebook does not claim that run occurred.

In [ ]:
import subprocess,os
env=os.environ.copy();env["PYTHONPATH"]=str(ROOT/"src")
r=subprocess.run([sys.executable,"-m","transaction_anomaly.cli","train","--help"],cwd=ROOT,env=env,text=True,capture_output=True)
print(r.stdout[:7000] or r.stderr[:3000])

## Extended study design — what should an anomaly detector accomplish?

Anomaly screening is a **ranking and workload management** problem: how many genuinely anomalous transactions appear among alerts, how many are missed, and how performance changes over time. Training must not learn information from transactions occurring after the prediction time. This notebook requires a verified real labeled CSV; **no synthetic fallback**.

In [ ]:
# Load real data for the extended analysis. Run after providing data/real/transactions.csv.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
real_path=ROOT/"data"/"real"/"transactions.csv"
if not real_path.is_file():
    raise FileNotFoundError("No authenticated real transaction CSV: skip analysis, not substitute synthetic data.")
data_real=pd.read_csv(real_path)
needed=["transaction_id","timestamp","amount","international","card_present","account_age_days","is_anomaly"]
missing=[c for c in needed if c not in data_real]
if missing: raise ValueError(f"Required real-data features missing: {missing}")
data_real["timestamp"]=pd.to_datetime(data_real["timestamp"],utc=True,errors="raise")
data_real=data_real.sort_values("timestamp")
display(data_real.head())
display(data_real[needed].isna().mean().rename("missing_fraction").to_frame())

## Class balance and transaction size

With rare anomalies, ordinary accuracy can reward a trivial all-normal predictor. Visualize outcomes and log-transformed transaction amounts, but avoid over-interpreting transaction value without knowing currency, refunds and data collection policy.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
data_real["is_anomaly"].value_counts().sort_index().plot.bar(ax=axes[0])
axes[0].set(title="Labeled outcomes (0 normal, 1 anomalous)",xlabel="Label",ylabel="Transactions")
for label,part in data_real.groupby("is_anomaly"):
    axes[1].hist(np.log1p(part["amount"].clip(lower=0)),bins=35,density=True,alpha=.5,label=f"Label {label}")
axes[1].set(title="Log-transformed transaction amount",xlabel="log(1 + amount)",ylabel="Density")
axes[1].legend();plt.tight_layout();plt.show()
print("Anomaly prevalence:",data_real["is_anomaly"].mean())

## Temporal drift, not random splitting

A useful screening evaluation trains on earlier dates, selects a model and alert threshold on later validation dates, and checks a final untouched future period. Choose dates based on the actual CSV coverage. Do not copy the synthetic example's fixed 2025 cutoffs.

In [ ]:
daily=(data_real.set_index("timestamp").resample("7D")["is_anomaly"]
       .agg(["mean","size"]).rename(columns={"mean":"anomaly_rate","size":"transactions"}))
fig,axes=plt.subplots(2,1,figsize=(11,6),sharex=True)
daily["transactions"].plot(ax=axes[0],title="Weekly transaction volume")
daily["anomaly_rate"].plot(ax=axes[1],title="Weekly labeled anomaly rate")
axes[1].set_ylabel("Observed prevalence");plt.tight_layout();plt.show()
print("Coverage:",data_real.timestamp.min(),"to",data_real.timestamp.max())

In [ ]:
from pathlib import Path
for name in ["data.py","pipeline.py","evaluation.py","cli.py"]:
    source=(ROOT/"src"/"transaction_anomaly"/name).read_text(encoding="utf-8")
    import ast
    tree=ast.parse(source)
    funcs=[n.name for n in tree.body if isinstance(n,(ast.FunctionDef,ast.AsyncFunctionDef,ast.ClassDef))]
    print(name,":",", ".join(funcs[:24]))

## Error and calibration report template

Only after training on the genuine dataset, examine the generated `metrics.json` and `test_errors.csv` inside a completed run directory. Do not paste numeric results into this notebook unless those files exist and the run's source hash and time cutoffs have been verified.

In [ ]:
# Example reporting cell: choose an existing genuine-data run explicitly.
from pathlib import Path
import json
RUN=ROOT/"runs"/"real-data-run"  # Replace with the actual completed run name.
metrics_path=RUN/"metrics.json"
if not metrics_path.exists():
    print("No verified real-data run metrics found; evaluation plots intentionally skipped.")
else:
    metric_result=json.loads(metrics_path.read_text(encoding="utf-8"))
    print("Reported sections:",list(metric_result) if isinstance(metric_result,dict) else type(metric_result))
    if (RUN/"test_errors.csv").exists():
        errors=pd.read_csv(RUN/"test_errors.csv")
        display(errors.head())
        print("Error records:",len(errors))

## Threats to validity and deployment decision

Check delayed fraud labels, multiple transactions from one account, charge reversals, data leakage via post-event fields, shifting merchants, region/currency mix, and alert capacity. Precision does not tell you the financial cost of an investigation; test-set performance is not a production SLA. **Status:** real dataset and executed real-data benchmark still required.

## Conclusion and next steps

The study remains limited to the data actually available. Complete real-data experiments and independently verified results are required before drawing deployment or domain-specific performance conclusions.

---

**Further development:** A separate Python implementation supports schema validation, temporal partitions, model selection, evaluation, and reproducible batch scoring.